#### Import thư viện

In [3]:
import pandas as pd
import numpy as np
import os
from collections import Counter
import joblib

from sklearn.preprocessing import MultiLabelBinarizer, OneHotEncoder

#### Đọc dữ liệu

In [ ]:
df = pd.read_csv('data/processed_movies.csv')
df.head()

,id,title,genres,original_language,overview,popularity,production_companies,release_date,budget,revenue,runtime,tagline,vote_average,vote_count,credits,keywords,poster_path,backdrop_path,recommendations
0,21972,Like Mike,Family-Comedy-Fantasy,en,Calvin and his friends who all live in an orph...,1.000000,20th Century Fox-Heller Highwater Productions-...,2002-07-03,0.447761,0.655650,0.456311,"Think like Mike, Achieve like Mike, Be Like Mike.",0.469388,0.272977,Shad Moss-Morris Chestnut-Jonathan Lipnicki-Br...,bet-lightning-sports-orphanage-bullying-basket...,/zXTM3qAXVYAYrhfYBpmescKMoes.jpg,/lh0uM88KjXsmRzWXUGAGWJ0AOJR.jpg,26655-203534-224303-10859-594404-14846-11374-1...
1,1687,Escape from the Planet of the Apes,Action-Science Fiction,en,The world is shocked by the appearance of thre...,0.999912,APJAC Productions-20th Century Fox,1971-05-20,0.037313,0.130499,0.446602,Meet baby Milo who has Washington terrified.,0.510204,0.506329,Roddy McDowall-Kim Hunter-Bradford Dillman-Nat...,spacecraft-dystopia-pacifism-politician-sequel...,/AnbLVdUEroTfHTUVAJCxkL4R0IH.jpg,/6JxJminJvYIVXbTakYJkbnga14P.jpg,1688-1705-1685-871-31067-869-811-2161-11234-18...
2,734002,The Peasants,Drama-History,pl,Peasant girl Jagna is forced to marry the much...,0.996262,Breakthru Films-ArtShot-DigitalKraft-Canal+ Po...,2023-10-13,0.111567,0.103253,0.611650,Love comes and goes but land stays.,0.816327,0.047881,Kamila Urzędowska-Robert Gulaczyk-Mirosław Bak...,adultery-based on novel or book-peasant-nobel ...,/vxwdArOG3R5AUHdvmwE4e7MLc0z.jpg,/rhFFbdRxSw25LvylJMbkOGljy2d.jpg,153917-84903-385152-1015622-1077670-118737-953...
3,11212,Baby's Day Out,Family-Comedy-Adventure-Crime-Drama,en,Baby Bink couldn't ask for more: he has adorin...,0.994202,20th Century Fox-Hughes Entertainment,1994-07-01,0.746269,0.175452,0.456311,No Bib. No Crib. No Problem.,0.510204,0.611447,Joe Mantegna-Lara Flynn Boyle-Joe Pantoliano-B...,chicago illinois-gorilla-baby-kidnapping-stupi...,/21U2jwl36hoTHsXB3fDuIQkcchu.jpg,/hq8dZfJEem1SIBCElMAuaJIfl6Q.jpg,361751-29918-332-10631-12139-1634-17414-11011-...
4,59429,Paranormal Activity: Tokyo Night,Horror-Thriller,ja,Koichi takes care of his sister who has recent...,0.992171,IM Global-Cinema Sunshine-Presidio-Musashino Ad,2010-11-20,0.020149,0.043780,0.378641,The nightmare continue...,0.448980,0.204183,Aoi Nakamura-Noriko Aoyama-Kosuke Kujirai-Ayak...,sequel-mockumentary-ghost-found footage-parano...,/mh7j7XlYWo82UnQGhsxcuyolzp.jpg,/1S6HTRZ8HwuIM8KVNOaycN5WxwX.jpg,41436-227348-72571-82990-146301-23827-609972-4...


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5895 entries, 0 to 5894
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    5895 non-null   int64  
 1   title                 5895 non-null   object 
 2   genres                5895 non-null   object 
 3   original_language     5895 non-null   object 
 4   overview              5881 non-null   object 
 5   popularity            5895 non-null   float64
 6   production_companies  5895 non-null   object 
 7   release_date          5895 non-null   object 
 8   budget                5895 non-null   float64
 9   revenue               5895 non-null   float64
 10  runtime               5895 non-null   float64
 11  tagline               4540 non-null   object 
 12  vote_average          5895 non-null   float64
 13  vote_count            5895 non-null   float64
 14  credits               5884 non-null   object 
 15  keywords             

### 1. Preprocessing

Bỏ bớt các feature ta không quan tâm

In [49]:
# Drop the column that we don't need
df.drop(columns=['overview', 'tagline', 'vote_average', 'vote_count', 'poster_path', 'backdrop_path', 'recommendations'], inplace=True)


Dùng cột `id` là index

In [50]:
df.set_index('id', inplace=True)

Xóa các dòng có dữ liệu thiếu

In [51]:
df = df.dropna()
df.shape

(5237, 11)

Tách nội dung của các cột `genres`, `keywords` thành list

In [52]:
df.loc[:, 'genres'] = df['genres'].apply(lambda x: x.split('-'))
df.loc[:, 'keywords'] = df['keywords'].apply(lambda x: x.split('-'))

Kiểm tra

In [53]:
print(df.info())
df.head(5)

<class 'pandas.core.frame.DataFrame'>
Index: 5237 entries, 21972 to 315226
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   title                 5237 non-null   object 
 1   genres                5237 non-null   object 
 2   original_language     5237 non-null   object 
 3   popularity            5237 non-null   float64
 4   production_companies  5237 non-null   object 
 5   release_date          5237 non-null   object 
 6   budget                5237 non-null   float64
 7   revenue               5237 non-null   float64
 8   runtime               5237 non-null   float64
 9   credits               5237 non-null   object 
 10  keywords              5237 non-null   object 
dtypes: float64(4), object(7)
memory usage: 491.0+ KB
None


,title,genres,original_language,popularity,production_companies,release_date,budget,revenue,runtime,credits,keywords
id,,,,,,,,,,,
21972,Like Mike,"[Family, Comedy, Fantasy]",en,1.000000,20th Century Fox-Heller Highwater Productions-...,2002-07-03,0.447761,0.655650,0.456311,Shad Moss-Morris Chestnut-Jonathan Lipnicki-Br...,"[bet, lightning, sports, orphanage, bullying, ..."
1687,Escape from the Planet of the Apes,"[Action, Science Fiction]",en,0.999912,APJAC Productions-20th Century Fox,1971-05-20,0.037313,0.130499,0.446602,Roddy McDowall-Kim Hunter-Bradford Dillman-Nat...,"[spacecraft, dystopia, pacifism, politician, s..."
734002,The Peasants,"[Drama, History]",pl,0.996262,Breakthru Films-ArtShot-DigitalKraft-Canal+ Po...,2023-10-13,0.111567,0.103253,0.611650,Kamila Urzędowska-Robert Gulaczyk-Mirosław Bak...,"[adultery, based on novel or book, peasant, no..."
11212,Baby's Day Out,"[Family, Comedy, Adventure, Crime, Drama]",en,0.994202,20th Century Fox-Hughes Entertainment,1994-07-01,0.746269,0.175452,0.456311,Joe Mantegna-Lara Flynn Boyle-Joe Pantoliano-B...,"[chicago illinois, gorilla, baby, kidnapping, ..."
59429,Paranormal Activity: Tokyo Night,"[Horror, Thriller]",ja,0.992171,IM Global-Cinema Sunshine-Presidio-Musashino Ad,2010-11-20,0.020149,0.043780,0.378641,Aoi Nakamura-Noriko Aoyama-Kosuke Kujirai-Ayak...,"[sequel, mockumentary, ghost, found footage, p..."


### 2. Xử lý data để chuẩn bị cho tìm kiếm cosine similarity

##### 2.1 Xử lý việc có quá nhiều category ở một số cột

Ở đây với các feature multi-label categorical có số category quá cao, ta sẽ chỉ giữ lại 2500 category phổ biến nhất

In [54]:
# Prunning multi label categorical column to keep top_n most frequent values
def prunning(df, feature, top_n=500):
    counter = Counter()
    for list_of_value in df[feature]:
        counter.update(list_of_value)
        
    most_common_categories = set(category for category, count in counter.most_common(top_n))
    df[feature] = df[feature].apply(lambda x: [category for category in x if category in most_common_categories])
    
    # Save the valid category to a file for later use
    with open(f'valid_{feature}.txt', 'w') as f:
        for category in most_common_categories:
            f.write(f'{category}\n')
    return df

# Apply prunning to the features
pruned_df = df.copy()
mlb_features = ['genres', 'keywords']
for feature in mlb_features:
    pruned_df = prunning(pruned_df, feature, top_n=500)

In [56]:
# Kiểm tra số category trong mỗi cột
mlb_features = ['genres', 'keywords']
for feature in mlb_features:
    counter = Counter()
    for list_of_value in pruned_df[feature]:
        counter.update(list_of_value)
    
    print(f"Feature: {feature}")
    print(f"Number of unique values: {len(counter)}\n")

Feature: genres
Number of unique values: 20

Feature: keywords
Number of unique values: 500



Xóa các dòng có giá trị rỗng (Do ta đã prunning bớt các category của một số cột)

In [57]:
mlb_features = ['genres', 'keywords']

for feature in mlb_features:
    pruned_df = pruned_df[pruned_df[feature].apply(lambda x: len(x) > 0)]
    
print(pruned_df.shape)

(4739, 11)


Đối với feature `original_language` ta sẽ chỉ lấy các bộ phim có ngôn ngữ thuộc top 15

In [58]:
counter = Counter(pruned_df['original_language'])
top_15_languages = set(language for language, count in counter.most_common(15))

# Discard all the film that are not in the top 15 languages
pruned_df = pruned_df[pruned_df['original_language'].isin(top_15_languages)]

# Save the valid category to a file for later use
with open('valid_original_language.txt', 'w') as f:
    for language in top_15_languages:
        f.write(f'{language}\n')

In [59]:
pruned_df.shape

(4597, 11)

##### 2.2 Tiến hành encode các đặc trưng multilabel categorical

Lưu ý: Dữ liệu sau khi encode sẽ tạo thành một dataframe mới chứ không ghi đè nên dataframe cũ

In [60]:
# Create binarizer
genres_binarizer = MultiLabelBinarizer()
keywords_binarizer = MultiLabelBinarizer()

# fit binarizer
genres_binarizer.fit(pruned_df['genres'])
keywords_binarizer.fit(pruned_df['keywords'])

# Save binarizer
joblib.dump(genres_binarizer, 'genres_binarizer.pkl')
joblib.dump(keywords_binarizer, 'keywords_binarizer.pkl')

# Transform
genres_encoded = genres_binarizer.transform(pruned_df['genres'])
keywords_encoded = keywords_binarizer.transform(pruned_df['keywords'])

# Create DataFrame
genres_encoded_df = pd.DataFrame(
    genres_encoded,
    columns=[f"genres_{name}" for name in genres_binarizer.classes_],
    index=pruned_df.index
)
keywords_encoded_df = pd.DataFrame(
    keywords_encoded,
    columns=[f"keywords_{name}" for name in keywords_binarizer.classes_],
    index=pruned_df.index
)

mlb_features_df = pd.concat([genres_encoded_df, keywords_encoded_df], axis=1)

##### 2.3 Tiến hành encode đặc trưng singlelabel categorical

In [62]:
language_encoder = OneHotEncoder()

# Fit the encoder
language_encoder.fit(pruned_df[['original_language']])

# Save the encoder
joblib.dump(language_encoder, 'language_encoder.pkl')

# Transform
language_encoded = language_encoder.transform(pruned_df[['original_language']])

# Create DataFrame
language_encoded_df = pd.DataFrame(
    language_encoded.toarray(),
    columns=[f"language_{name}" for name in language_encoder.categories_[0]],
    index=pruned_df.index
)

##### 2.3 Tạo bản dataframe cuối cùng

In [64]:
# Final DataFrame
similarity_df = pd.concat([
    pruned_df['budget'],
    mlb_features_df,
    language_encoded_df,
], axis=1)

In [65]:
similarity_df

,budget,genres_Action,genres_Adventure,genres_Animation,genres_Comedy,genres_Crime,genres_Documentary,genres_Drama,genres_Family,genres_Fantasy,...,language_hi,language_it,language_ja,language_ko,language_ml,language_no,language_ru,language_sv,language_ta,language_zh
id,,,,,,,,,,,,,,,,,,,,,
21972,4.477612e-01,0,0,0,1,0,0,0,1,1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1687,3.731342e-02,1,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
11212,7.462687e-01,0,1,0,1,1,0,1,1,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
59429,2.014924e-02,0,0,0,0,0,0,0,0,0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
60747,9.701493e-01,1,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
587914,1.195373e-04,1,0,0,0,0,0,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
598899,4.968657e-05,1,0,0,0,0,0,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
285908,5.671640e-02,0,0,0,0,1,0,1,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [66]:
# Except the budget column, turn all column into int
for column in similarity_df.columns:
    if column != 'budget':
        similarity_df[column] = similarity_df[column].astype(int)
        
similarity_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4597 entries, 21972 to 325453
Columns: 536 entries, budget to language_zh
dtypes: float64(1), int64(535)
memory usage: 18.8 MB


In [ ]:
# Save the similarity_df to a CSV file
similarity_df.to_csv('data/similarity.csv', index=True)